In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the CSV file
import os
import pandas as pd
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)


In [ ]:
# Task 2: Write your code here:
# inspecting the shape and first few raws of the datasetrows of the dataset
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Task 3: Write your code here:
# Check data info
df.info()

In [ ]:
# Task 4: Write your code here:
# Descriptive statistics for numerical columns
df.describe()

In [ ]:
# Task 5: Write your code here:
# plotting the target variable (delivery time)
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

# Drop 'Order_ID' Columns with drop
df_clean = df.drop(columns='Order_ID').copy()
print(f"Shape after cleaning: {df_clean.shape}")
df_clean.head()

In [ ]:
# Task 2: Write your code here:
# Do we have missing values?
# defining a function to check the missing values
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nWe Need to Handle Missing Values.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

print("Separating target variable and features...")
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Applying One-Hot Encoding to feature DataFrame 'X'...")
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))

print("Verification of encoded data shapes:")
print(f"Shape of X_encoded: {X_encoded.shape}")
print(f"Shape of y_encoded: {y_encoded.shape}")

print("First 5 rows of X_encoded:")
display(X_encoded.head())
print("First 5 elements of y_encoded:")
print(y_encoded[:5])

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols =  df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# TODO: Apply fit_transform to scale the numerical columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
# Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df['Delivery_Time'].value_counts(normalize=True))
  sns.countplot(x=df['Delivery_Time'])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "status")

In [ ]:
# Task 1: Write your code here:
print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.3, random_state=42)

print("Data split successful.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
# Categorical cross entropy loss function using NumPy
def categorical_cross_entropy(y, y_hat):
  epsilon = 1e-15
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon)

  loss = -np.mean(np.sum(y * np.log(y_hat), axis=1))
  return loss

In [ ]:
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")
print(f"RMSE: {rmse_scores.mean():,.2f}")

In [ ]:
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather',	'Traffic_Level'	,'Time_of_Day'	,'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here: